# 02_cuVS / 00 — Local multimodal embeddings with EmbeddingGemma 2 (transformers)

One open model turns **text, images, audio and video** into vectors in the **same** 768-dim space, so a text query can find a photo, a sound or a video clip. Everything runs on the local GPU with Hugging Face **transformers**: no API keys, no cloud storage.

| | |
|---|---|
| Model | [`google/embeddinggemma-2`](https://huggingface.co/google/embeddinggemma-2): Apache 2.0, not gated, 740M params |
| Output | 768 dims, L2-normalized (MRL: also 512 / 256 / 128) |
| Context | 8,192 tokens. Image ≈ 280 tokens, video ≈ 140 per frame at 1 fps, audio ≈ 25 per second (16 kHz mono) |
| Inference | `transformers`: `AutoProcessor` + `AutoModel`, then mean pooling and L2 normalization |

**What this notebook does**
1. Install, then check the GPU
2. Load the processor and model
3. Cross-modal search check on sample images, sounds and videos
4. Text search, mixed (text + image) inputs, MRL truncation
5. Throughput per modality, saved to `results/`

**On a T4 the model runs in float32.** The model card warns that float16 gives NaN or silently wrong vectors, and the T4 has no native bfloat16. Newer GPUs (L4, A100, …) load in bfloat16 automatically.

## 1. Install

`google/embeddinggemma-2` is not gated, so no Hugging Face token is needed. (A token only raises download rate limits: set `os.environ["HF_TOKEN"]` before section 3 if you have one, and never commit it.)

Restart the kernel after installing if `transformers` was upgraded.

In [ ]:
!pip install -q -U transformers accelerate librosa soundfile opencv-python-headless scikit-image

## 2. Environment

In [ ]:
import os, gc, json, time, platform
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import transformers

props = torch.cuda.get_device_properties(0)
cc = torch.cuda.get_device_capability(0)
env = {
    "python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "gpu": props.name, "gpu_mem_gb": round(props.total_memory / 1e9, 1),
    "compute_capability": f"{cc[0]}.{cc[1]}", "cpu_cores": os.cpu_count(),
}
pd.Series(env, name="value").to_frame()

## 3. The embedder

**This cell is copied into every notebook in this folder**, so each notebook runs inference itself.

```python
processor = AutoProcessor.from_pretrained("google/embeddinggemma-2")
model     = AutoModel.from_pretrained("google/embeddinggemma-2", device_map="auto", dtype=DTYPE)
```

`model(...)` returns one 768-dim vector **per token** (`last_hidden_state`). The checkpoint's own pipeline (`modules.json`) turns that into one vector per item: **mean pooling** over all tokens, then **L2 normalization**. That is also why we don't use `pipeline("feature-extraction")`: it returns the per-token vectors and handles text only.

Items are plain dicts. Text carries the model's task prefix; media uses a placeholder token and no prefix:

```python
query_item("what causes the aurora")         # {"text": "task: search result | query: ..."}
doc_item("The aurora is ...", title="Aurora")  # {"text": "title: Aurora | text: ..."}
image_item(pil_image)   audio_item("clip.wav")   video_item("clip.mp4")
{"text": "Red shoes <|image|> on wet rock <|video|>", "image": img, "video": "demo.mp4"}   # mixed
```

For speed:
- Audio (resampled to 16 kHz mono) and video (OpenCV) are **decoded on all CPU cores** in parallel.
- Items are **grouped by media type** and **sorted by length**, so each batch has little padding.
- Inference runs under `torch.inference_mode()`.

In [ ]:
from transformers import AutoModel, AutoProcessor

MODEL_ID = "google/embeddinggemma-2"
DIM      = 768
DTYPE    = torch.bfloat16 if torch.cuda.get_device_capability(0) >= (8, 0) else torch.float32   # never float16
BATCH    = {"text": 32, "image": 32, "audio": 16, "video": 4}     # batch size per modality
MEDIA    = ("image", "audio", "video")

def query_item(q):                return {"text": f"task: search result | query: {q}"}
def doc_item(text, title="none"): return {"text": f"title: {title} | text: {text}"}
def image_item(img):              return {"text": "<|image|>", "image": img}
def audio_item(path):             return {"text": "<|audio|>", "audio": path}
def video_item(path):             return {"text": "<|video|>", "video": path}

def kinds(item):
    return tuple(m for m in MEDIA if m in item) or ("text",)

def normalize(x, dim=DIM):
    """Truncate to `dim` (MRL) and L2-normalize."""
    x = np.asarray(x, dtype=np.float32)[:, :dim]
    return x / np.linalg.norm(x, axis=1, keepdims=True)

def _decode(item):
    out = dict(item)
    if "audio" in item:
        import librosa
        out["audio"] = librosa.load(item["audio"], sr=16_000, mono=True)[0]
    if "video" in item:
        from transformers.video_utils import load_video
        out["video"] = load_video(item["video"], backend="opencv")          # (frames, metadata)
    return out

def decode_all(items):
    """Load audio/video into memory on all CPU cores."""
    if not any("audio" in it or "video" in it for it in items):
        return list(items)
    with ThreadPoolExecutor(os.cpu_count()) as pool:
        return list(pool.map(_decode, items))


class Embedder:
    """google/embeddinggemma-2 with Hugging Face transformers."""

    def __init__(self):
        t0 = time.perf_counter()
        self.processor = AutoProcessor.from_pretrained(MODEL_ID)
        self.model = AutoModel.from_pretrained(MODEL_ID, device_map="auto", dtype=DTYPE).eval()
        self.device = self.model.device
        n = sum(p.numel() for p in self.model.parameters()) / 1e6
        print(f"{MODEL_ID}: {n:.0f}M params, {str(DTYPE).replace('torch.', '')}, "
              f"{torch.cuda.memory_allocated() / 1e9:.2f} GB GPU, loaded in {time.perf_counter() - t0:.0f} s")

    @torch.inference_mode()
    def _forward(self, items):
        """One batch of items that share the same media types -> (batch, 768) mean-pooled vectors."""
        kw = {"text": [it["text"] for it in items], "return_tensors": "pt", "padding": True}
        if "image" in items[0]:
            kw["images"] = [it["image"] if isinstance(it["image"], list) else [it["image"]] for it in items]
        if "audio" in items[0]:
            kw["audio"], kw["sampling_rate"] = [it["audio"] for it in items], 16_000
        if "video" in items[0]:
            kw["videos"], kw["video_metadata"] = [it["video"][0] for it in items], [it["video"][1] for it in items]
        batch = {k: v.to(self.device, DTYPE) if v.is_floating_point() else v.to(self.device)
                 for k, v in self.processor(**kw).items() if torch.is_tensor(v)}
        hidden = self.model(**batch).last_hidden_state                    # (batch, tokens, 768)
        mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        return ((hidden * mask).sum(1) / mask.sum(1)).float().cpu().numpy()

    def embed(self, items, dim=DIM):
        """List of item dicts -> float32 array (len(items), dim), L2-normalized."""
        out = np.zeros((len(items), DIM), np.float32)
        groups = {}
        for i, it in enumerate(items):
            groups.setdefault(kinds(it), []).append(i)
        for kind, idx in groups.items():
            idx.sort(key=lambda i: len(items[i]["text"]))                 # similar lengths -> less padding
            bs = BATCH[kind[0]]
            for j in range(0, len(idx), bs):
                part = idx[j:j + bs]
                out[part] = self._forward(decode_all([items[i] for i in part]))
        return normalize(out, dim)

    def close(self):
        self.model = None
        gc.collect(); torch.cuda.empty_cache()


emb = Embedder()

## 4. Sample media

Small local samples (the real datasets come in notebook 01):
- **Images:** four photos bundled with scikit-image
- **Audio:** four 15 s clips from librosa's examples: whale song, trumpet, speech, drums and bass
- **Video:** two 8 s clips generated here: a bouncing red ball, and a slow zoom into the cat photo

In [ ]:
import cv2, soundfile as sf, librosa
from PIL import Image
from skimage import data as sk

os.makedirs("assets", exist_ok=True)

images = {"an astronaut": sk.astronaut(), "a cat": sk.chelsea(), "a cup of coffee": sk.coffee(), "a rocket": sk.rocket()}
images = {k: Image.fromarray(v) for k, v in images.items()}

audios = {}
for key, label in {"humpback": "whale song", "trumpet": "trumpet", "libri1": "speech", "choice": "drums and bass"}.items():
    path = f"assets/{key}.wav"
    if not os.path.exists(path):
        sf.write(path, librosa.load(librosa.ex(key), sr=16_000, mono=True, duration=15)[0], 16_000)
    audios[label] = path

def write_video(path, frames, fps=10):
    h, w = frames[0].shape[:2]
    vw = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    for f in frames:
        vw.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
    vw.release()

if not os.path.exists("assets/red_ball.mp4"):
    frames = []
    for t in range(80):
        f = np.full((240, 320, 3), 235, np.uint8)
        cv2.line(f, (0, 220), (320, 220), (90, 90, 90), 3)
        cv2.circle(f, (40 + 3 * t, int(200 - 160 * abs(np.sin(t / 8)))), 18, (220, 30, 30), -1)
        frames.append(f)
    write_video("assets/red_ball.mp4", frames)
if not os.path.exists("assets/cat_zoom.mp4"):
    cat, frames = sk.chelsea(), []
    H, W = cat.shape[:2]
    for t in range(80):
        s = 1 - 0.5 * t / 80
        h, w = int(H * s), int(W * s)
        frames.append(cv2.resize(cat[(H - h) // 2:(H + h) // 2, (W - w) // 2:(W + w) // 2], (320, 240)))
    write_video("assets/cat_zoom.mp4", frames)
videos = {"a red ball bouncing": "assets/red_ball.mp4", "a cat (zooming in)": "assets/cat_zoom.mp4"}

print(f"{len(images)} images, {len(audios)} audio clips, {len(videos)} videos")

## 5. Cross-modal search

Ten **text** queries against one pool of 10 media items (4 images, 4 sounds, 2 videos). Each query has exactly one correct item, in a different modality.

In [ ]:
pool   = [image_item(v) for v in images.values()] + [audio_item(v) for v in audios.values()] + [video_item(v) for v in videos.values()]
labels = [f"[image] {k}" for k in images] + [f"[audio] {k}" for k in audios] + [f"[video] {k}" for k in videos]
queries = ["an astronaut in a space suit", "a cat", "a cup of coffee", "a rocket launch",
           "a whale singing underwater", "a trumpet playing", "a person reading a book aloud",
           "drums and bass music", "a red ball bouncing on the floor", "a close-up of a cat"]

t0 = time.perf_counter()
P = emb.embed(pool)
Q = emb.embed([query_item(q) for q in queries])
print(f"embedded {len(pool)} media items + {len(queries)} queries in {time.perf_counter() - t0:.1f} s")

S = Q @ P.T
top1 = int((S.argmax(1) == np.arange(len(queries))).sum())
print(f"top-1 correct: {top1} / {len(queries)}\n")
for i, (q, row) in enumerate(zip(queries, S)):
    print(f"{'ok' if row.argmax() == i else '--'}  {q:<34} -> {labels[row.argmax()]:<30} {row.max():.3f}")

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(S, cmap="viridis")
ax.set_xticks(range(len(labels)), labels, rotation=40, ha="right", fontsize=8)
ax.set_yticks(range(len(queries)), queries, fontsize=8)
for i, j in enumerate(S.argmax(1)):
    ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec="white", lw=2))
fig.colorbar(im, ax=ax, label="cosine similarity")
ax.set_title(f"Text queries vs images, audio and video  (top-1 {top1}/{len(queries)})")
plt.tight_layout(); plt.show()

## 6. Text search, mixed inputs and MRL

In [ ]:
docs = [("Aurora", "The northern lights are caused by charged particles from the sun colliding with gases in the atmosphere."),
        ("Turing (microarchitecture)", "Turing is an NVIDIA GPU architecture used in the Tesla T4 and the GeForce RTX 20 series."),
        ("Mango", "The mango is an edible stone fruit produced by the tropical tree Mangifera indica.")]
text_q = ["what causes the aurora borealis", "which gpu architecture does the T4 use", "tropical fruit"]
D = emb.embed([doc_item(t, title) for title, t in docs])
print("Text search (rows: queries, cols: docs)")
print(pd.DataFrame(emb.embed([query_item(q) for q in text_q]) @ D.T, index=text_q, columns=[d[0] for d in docs]).round(3), "\n")

mixed = emb.embed([{"text": "Space mission photo: <|image|> crew member ready for launch", "image": images["an astronaut"]},
                   {"text": "Breakfast table: <|image|> freshly brewed", "image": images["a cup of coffee"]}])
mix_q = ["astronaut preparing for a space launch", "morning coffee"]
print("Mixed text + image items")
print(pd.DataFrame(emb.embed([query_item(q) for q in mix_q]) @ mixed.T, index=mix_q, columns=["astronaut post", "coffee post"]).round(3), "\n")

print("MRL: shorter vectors (re-normalized), same top-1 as 768 dims?")
for d in [512, 256, 128]:
    agree = int(((normalize(Q, d) @ normalize(P, d).T).argmax(1) == S.argmax(1)).sum())
    print(f"  {d:>3} dims: {agree}/{len(queries)} queries agree, {d * 4:>4} bytes per vector ({768 / d:.1f}x smaller)")

## 7. Throughput

Items per second for each modality on this GPU (one warmup, then a timed run). Text uses ~500-token passages, the chunk size planned for notebook 01.

In [ ]:
passage = " ".join(("Graphics processors run thousands of threads in parallel, which makes them well suited to "
                    "embedding large collections of documents, images, audio and video. ").split() * 20)[:2600]
bench = {
    "text (500 tok)": [doc_item(f"{passage} {i}", "Benchmark") for i in range(128)],
    "image":          [image_item(list(images.values())[i % 4]) for i in range(64)],
    "audio (15 s)":   [audio_item(list(audios.values())[i % 4]) for i in range(32)],
    "video (8 s)":    [video_item(list(videos.values())[i % 2]) for i in range(16)],
}

def throughput(items):
    emb.embed(items[:4]); torch.cuda.synchronize()                       # warmup
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter(); emb.embed(items); torch.cuda.synchronize()
    return len(items) / (time.perf_counter() - t0), torch.cuda.max_memory_allocated() / 1e9

rows = {k: throughput(v) for k, v in bench.items()}
tp = pd.DataFrame(rows, index=["items / s", "peak GPU GB"]).T
tp["items / hour"] = (tp["items / s"] * 3600).astype(int)
display(tp.round(2))

ax = tp["items / s"].plot.barh(figsize=(8, 2.8), logx=True, color="#4C9F70")
ax.set_xlabel("items per second (log, higher is better)")
ax.set_title(f"EmbeddingGemma 2 on {props.name} ({str(DTYPE).replace('torch.', '')})")
for i, v in enumerate(tp["items / s"]):
    ax.text(v, i, f" {v:.1f}/s", va="center", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
os.makedirs("results", exist_ok=True)
report = {"env": env, "dtype": str(DTYPE), "cross_modal_top1": f"{top1}/{len(queries)}",
          "throughput": tp.round(3).to_dict(orient="index")}
with open("results/00_embedder_check.json", "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps({k: report[k] for k in ["dtype", "cross_modal_top1"]}, indent=2))

## Summary

- **EmbeddingGemma 2 runs locally with transformers** and puts text, images, audio and video in one 768-dim space. Section 5 checks that a text query lands on the right image, sound or video.
- **The embedder cell** (section 3) is copied into every notebook in this folder.
- **MRL.** Vectors can be cut to 512 / 256 / 128 dims (re-normalized) to save memory; notebooks 03–06 measure what that does to search quality and speed.
- **Throughput** (items per hour) sets how large a corpus notebook 01 can embed in reasonable time.

**Next:** `01_build_multimodal_corpus.ipynb` — real datasets for every modality, embedded and saved locally.